# Gold Layer — Exploratory Data Analysis (EDA)

This notebook loads the three gold-layer views exported as CSV files from the
`exports/gold/` folder and performs an end-to-end exploratory analysis:

| # | Section |
|---|---------|
| 1 | Setup & imports |
| 2 | Load data |
| 3 | Data quality inspection |
| 4 | Sales trends over time |
| 5 | Product & category analysis |
| 6 | Customer demographics |
| 7 | Key summary metrics |

> **Before running:** make sure the `sql_medallion` conda environment is active and
> that `export_gold_views.py` has been executed so the CSVs exist.

## 1 — Setup & Imports

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Notebook display settings ─────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

# ── Path to exported CSVs ─────────────────────────────────────
# Resolves to  <repo_root>/exports/gold/  regardless of where
# JupyterLab is launched from.
EXPORTS = Path("..") / "exports" / "gold"

print("Environment ready ✓")

## 2 — Load Data

In [ ]:
# Parse date columns on load so we can use them directly for time series.
customers = pd.read_csv(
    EXPORTS / "dim_customers.csv",
    parse_dates=["birthdate", "create_date"],
)

products = pd.read_csv(EXPORTS / "dim_products.csv", parse_dates=["start_date"])

sales = pd.read_csv(
    EXPORTS / "fact_sales.csv",
    parse_dates=["order_date", "shipping_date", "due_date"],
)

print(f"dim_customers : {customers.shape[0]:>6,} rows × {customers.shape[1]} cols")
print(f"dim_products  : {products.shape[0]:>6,} rows × {products.shape[1]} cols")
print(f"fact_sales    : {sales.shape[0]:>6,} rows × {sales.shape[1]} cols")

## 3 — Data Quality Inspection

In [ ]:
def quality_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Return a tidy per-column summary: dtype, # nulls, % nulls, # unique values."""
    report = pd.DataFrame({
        "dtype":    df.dtypes.astype(str),
        "nulls":    df.isna().sum(),
        "null_%":   (df.isna().mean() * 100).round(2),
        "unique":   df.nunique(),
    })
    print(f"\n{'─'*50}")
    print(f"  {name}  ({df.shape[0]:,} rows × {df.shape[1]} cols)")
    print(f"{'─'*50}")
    return report

display(quality_report(customers,  "dim_customers"))
display(quality_report(products,   "dim_products"))
display(quality_report(sales,      "fact_sales"))

In [ ]:
# Quick peek at the first rows of each table
print("── dim_customers ──")
display(customers.head(3))

print("── dim_products ──")
display(products.head(3))

print("── fact_sales ──")
display(sales.head(3))

## 4 — Sales Trends Over Time

In [ ]:
# ── Monthly revenue trend ─────────────────────────────────────
monthly = (
    sales
    .assign(month=sales["order_date"].dt.to_period("M"))
    .groupby("month", as_index=False)["sales_amount"]
    .sum()
)
monthly["month_dt"] = monthly["month"].dt.to_timestamp()

fig, ax = plt.subplots()
ax.plot(monthly["month_dt"], monthly["sales_amount"], marker="o", linewidth=1.5)
ax.set_title("Monthly Revenue")
ax.set_xlabel("Month")
ax.set_ylabel("Sales Amount")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# ── Orders and quantity sold per month (secondary analysis) ───
monthly_qty = (
    sales
    .assign(month=sales["order_date"].dt.to_period("M"))
    .groupby("month", as_index=False)
    .agg(orders=("order_number", "nunique"), units_sold=("quantity", "sum"))
)
monthly_qty["month_dt"] = monthly_qty["month"].dt.to_timestamp()

fig, (ax1, ax2) = plt.subplots(1, 2)

ax1.bar(monthly_qty["month_dt"], monthly_qty["orders"], width=20, color="steelblue")
ax1.set_title("Unique Orders per Month")
ax1.set_ylabel("Orders")
fig.autofmt_xdate()

ax2.bar(monthly_qty["month_dt"], monthly_qty["units_sold"], width=20, color="coral")
ax2.set_title("Units Sold per Month")
ax2.set_ylabel("Quantity")
fig.autofmt_xdate()

plt.tight_layout()
plt.show()

## 5 — Product & Category Analysis

In [ ]:
# ── Revenue by product category ───────────────────────────────
# Join sales → products to get the category name.
sales_prod = sales.merge(
    products[["product_key", "product_name", "category", "subcategory", "cost"]],
    on="product_key",
    how="left",
)

cat_revenue = (
    sales_prod
    .groupby("category", as_index=False)["sales_amount"]
    .sum()
    .sort_values("sales_amount", ascending=False)
)

fig, ax = plt.subplots()
sns.barplot(data=cat_revenue, x="sales_amount", y="category", ax=ax, orient="h")
ax.set_title("Revenue by Category")
ax.set_xlabel("Total Sales Amount")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 10 products by revenue ────────────────────────────────
top_products = (
    sales_prod
    .groupby("product_name", as_index=False)["sales_amount"]
    .sum()
    .nlargest(10, "sales_amount")
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=top_products, x="sales_amount", y="product_name", ax=ax, orient="h")
ax.set_title("Top 10 Products by Revenue")
ax.set_xlabel("Total Sales Amount")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.show()

## 6 — Customer Demographics

In [ ]:
# ── Gender & marital status distribution ──────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2)

gender_counts = customers["gender"].value_counts()
ax1.pie(gender_counts, labels=gender_counts.index, autopct="%1.1f%%", startangle=90)
ax1.set_title("Gender Distribution")

marital_counts = customers["marital_status"].value_counts()
ax2.pie(marital_counts, labels=marital_counts.index, autopct="%1.1f%%", startangle=90)
ax2.set_title("Marital Status")

plt.tight_layout()
plt.show()

In [ ]:
# ── Customer count by country ─────────────────────────────────
country_counts = (
    customers["country"]
    .value_counts()
    .reset_index()
    .rename(columns={"count": "customers"})
)

fig, ax = plt.subplots()
sns.barplot(data=country_counts, x="count", y="country", ax=ax, orient="h")
ax.set_title("Customers by Country")
ax.set_xlabel("Number of Customers")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# ── Customer age distribution (derived from birthdate) ────────
customers_with_age = customers.dropna(subset=["birthdate"]).copy()
customers_with_age["age"] = (
    pd.Timestamp("today") - customers_with_age["birthdate"]
).dt.days // 365

fig, ax = plt.subplots()
ax.hist(customers_with_age["age"], bins=20, color="steelblue", edgecolor="white")
ax.set_title("Customer Age Distribution")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

print(customers_with_age["age"].describe().rename("age_stats"))

## 7 — Key Summary Metrics

In [ ]:
# ── High-level KPIs ───────────────────────────────────────────
total_revenue   = sales["sales_amount"].sum()
total_orders    = sales["order_number"].nunique()
avg_order_value = total_revenue / total_orders if total_orders else 0
total_customers = customers["customer_id"].nunique()
total_products  = products["product_id"].nunique()
date_range      = f"{sales['order_date'].min().date()}  →  {sales['order_date'].max().date()}"

kpis = pd.DataFrame({
    "Metric": [
        "Total Revenue",
        "Total Orders",
        "Avg Order Value",
        "Unique Customers",
        "Active Products",
        "Date Range",
    ],
    "Value": [
        f"${total_revenue:,.2f}",
        f"{total_orders:,}",
        f"${avg_order_value:,.2f}",
        f"{total_customers:,}",
        f"{total_products:,}",
        date_range,
    ],
})

kpis = kpis.set_index("Metric")
display(kpis.style.set_caption("Gold Layer — Summary KPIs"))